# Create Delta table and test Primary constraint

- The source comes from jupyter-pyspark/f1-sourcefiles
- Create delta lake table with primary key
- Import circuits.csv file into dataframe
- Add an extra row to break primary key contraint
- Insert data into delta table


# Initalise a spark session

In [1]:
# Initalise a spark session
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType,
    DoubleType, DateType, BooleanType
)
from pyspark.sql.functions import col
from delta.tables import DeltaTable

# Fix JAVA_HOME to your actual Java 21 path
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages io.delta:delta-spark_2.12:3.2.0 pyspark-shell"

# Build Spark session with Delta Lake support
builder = SparkSession.builder \
    .appName("DeltaLakeExample") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()


26/05/06 23:13:03 WARN Utils: Your hostname, DESKTOP-OQT8U26 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/06 23:13:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/robyip/projects/pyspark-deltalake/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/robyip/.ivy2/cache
The jars for the packages stored in: /home/robyip/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-36021e1f-e6df-48ce-a60f-a6ccd5eca8f0;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 312ms :: artifacts dl 8ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0 

# Setup schema (database)


In [4]:
spark.sql("CREATE DATABASE IF NOT EXISTS f1 COMMENT 'f1 schema'")
spark.sql("USE f1")


# races = spark.read.csv("f1-sourcefiles/races.csv", header=True, inferSchema=True).filter("year >= 2022")


# seasons = spark.read.csv("f1-sourcefiles/seasons.csv", header=True, inferSchema=True)

DataFrame[]

# Drop table if exists


In [31]:
spark.sql("DROP TABLE IF EXISTS f1.circuits")




DataFrame[]

# Use Alias in a RIGHT JOIN


In [32]:
# See where Spark is running from
print(f"Working directory: {os.getcwd()}")

# build an absolute path to it 

base_path  = os.getcwd()
table_path = f"{base_path}/delta/f1"

print(f"Writing to: {table_path}")


Working directory: /home/robyip/projects/pyspark-deltalake/jupyter-pyspark
Writing to: /home/robyip/projects/pyspark-deltalake/jupyter-pyspark/delta/f1


In [33]:
spark.sql(f"""
    CREATE TABLE f1.circuits (
        circuitId   INT       NOT NULL,
        circuitRef  STRING    NOT NULL,
        name        STRING    NOT NULL,
        location    STRING    NOT NULL,
        country     STRING,
        lat         STRING,
        lng         STRING,
        alt         INT       NOT NULL,
        url         STRING
    )
    USING DELTA
    LOCATION '{table_path}'
    COMMENT 'F1 circuits table'
    TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact'   = 'true'
    )
""")

DataFrame[]